# Gem Finder — OSM POI Scraper
Scrape Manhattan attraction data via the Overpass API

**Run once — results are written to `attractions_raw.json` and `attractions_final.csv`**

Data source: OpenStreetMap via Overpass API  
Coverage types: museum / gallery / attraction / viewpoint / aquarium / zoo / park / garden / theatre / cinema / arts_centre / historic / place_of_worship


## Section 1 — Install Dependencies

In [ ]:

# !pip install requests pandas tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)


## Section 2 — Configuration

In [ ]:
import json
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ── Output directory ────────────────────────────────────────────
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Overpass API endpoints (any one that responds is fine) ───────
OVERPASS_ENDPOINTS = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://maps.mail.ru/osm/tools/overpass/api/interpreter',
]

# ── Manhattan bounding box (covers Manhattan Island + Roosevelt Island) ──
# south, west, north, east
MANHATTAN_BBOX = (40.6960, -74.0210, 40.8820, -73.9070)

# ── Delay between requests (avoid Overpass rate limits) ─────────
REQUEST_DELAY_SEC = 3

print(f'Output dir: {OUTPUT_DIR.resolve()}')
print(f'Bounding box: {MANHATTAN_BBOX}')


Output dir: /Users/rj/Desktop/dev/code/COMP47360 /Data/output
Bounding box: (40.696, -74.021, 40.882, -73.907)


## Section 3 — Define Overpass Queries

In [ ]:
S, W, N, E = MANHATTAN_BBOX
BBOX_STR = f'{S},{W},{N},{E}'

# ── Define each tag group separately for easy adjustment ────────
#
# Format: (group_name, overpass_filter)
# group_name is used for logging only — not written to final data
#
TAG_GROUPS = [
    # ── tourism=* ──────────────────────────────────────
    ('museum',         '["tourism"="museum"]'),
    ('gallery',        '["tourism"="gallery"]'),
    ('aquarium',       '["tourism"="aquarium"]'),
    ('zoo',            '["tourism"="zoo"]'),
    ('attraction',     '["tourism"="attraction"]'),
    ('viewpoint',      '["tourism"="viewpoint"]'),
    ('artwork',        '["tourism"="artwork"]'),
    ('theme_park',     '["tourism"="theme_park"]'),

    # ── leisure=* ──────────────────────────────────────
    ('park',           '["leisure"="park"]'),
    ('garden',         '["leisure"="garden"]'),
    ('nature_reserve', '["leisure"="nature_reserve"]'),
    ('stadium',        '["leisure"="stadium"]'),

    # ── amenity=* ──────────────────────────────────────
    ('theatre',        '["amenity"="theatre"]'),
    ('cinema',         '["amenity"="cinema"]'),
    ('arts_centre',    '["amenity"="arts_centre"]'),
    ('place_of_worship', '["amenity"="place_of_worship"]["name"]'),  # named only

    # ── historic=* ─────────────────────────────────────
    ('monument',       '["historic"="monument"]'),
    ('memorial',       '["historic"="memorial"]'),
    ('historic_building', '["historic"="building"]'),
    ('landmark',       '["historic"="landmark"]'),
]

def build_query(tag_filter: str, bbox: str) -> str:
    """Build an Overpass QL query that fetches both nodes and ways
    (ways use 'out center' to return a centroid coordinate)."""
    return f"""
[out:json][timeout:60];
(
  node{tag_filter}({bbox});
  way{tag_filter}({bbox});
  relation{tag_filter}({bbox});
);
out body center;
"""

print(f'Defined {len(TAG_GROUPS)} tag groups to query')
for name, f in TAG_GROUPS:
    print(f'  {name:25s} {f}')


Defined 20 tag groups to query
  museum                    ["tourism"="museum"]
  gallery                   ["tourism"="gallery"]
  aquarium                  ["tourism"="aquarium"]
  zoo                       ["tourism"="zoo"]
  attraction                ["tourism"="attraction"]
  viewpoint                 ["tourism"="viewpoint"]
  artwork                   ["tourism"="artwork"]
  theme_park                ["tourism"="theme_park"]
  park                      ["leisure"="park"]
  garden                    ["leisure"="garden"]
  nature_reserve            ["leisure"="nature_reserve"]
  stadium                   ["leisure"="stadium"]
  theatre                   ["amenity"="theatre"]
  cinema                    ["amenity"="cinema"]
  arts_centre               ["amenity"="arts_centre"]
  place_of_worship          ["amenity"="place_of_worship"]["name"]
  monument                  ["historic"="monument"]
  memorial                  ["historic"="memorial"]
  historic_building         ["historic

## Section 4 — Request Helper Functions

In [ ]:
import random

def overpass_request(query: str, retries: int = 3) -> list:
    """
    Send a request to the Overpass API, randomising endpoint order
    to avoid hammering one server. Returns a list of elements;
    returns an empty list if all retries are exhausted.
    """
    endpoints = OVERPASS_ENDPOINTS.copy()
    random.shuffle(endpoints)  # shuffle to spread load

    for attempt, endpoint in enumerate(endpoints[:retries]):
        try:
            resp = requests.post(
                endpoint,
                data={'data': query},
                timeout=90,
                headers={'User-Agent': 'GemFinder-Research/1.0'}
            )
            if resp.status_code == 200:
                data = resp.json()
                return data.get('elements', [])
            elif resp.status_code in (429, 503, 504):
                wait = 10 * (attempt + 1)   # wait 10s / 20s / 30s
                print(f'  Server busy ({resp.status_code}) on {endpoint}, waiting {wait}s...')
                time.sleep(wait)
            else:
                print(f'  HTTP {resp.status_code} from {endpoint}, retrying...')
                time.sleep(5)
        except requests.exceptions.Timeout:
            print(f'  Timeout on {endpoint} (attempt {attempt+1}), retrying...')
            time.sleep(5)
        except Exception as e:
            print(f'  Error: {e}, retrying...')
            time.sleep(5)

    print(f'  ⚠️ All {retries} attempts failed, returning empty list')
    return []


print('Helper functions defined ✓')


Helper functions defined ✓


In [ ]:
def extract_center(el: dict):
    """
    Extract the centre coordinate from an Overpass element.
    - node type: lat/lon are direct fields on the element
    - way / relation type: the query used 'out center', so coords
      live inside el['center']
    Returns (None, None) if extraction fails.
    """
    if el['type'] == 'node':
        return el.get('lat'), el.get('lon')
    else:
        # ways and relations return a centroid via 'out center'
        center = el.get('center', {})
        return center.get('lat'), center.get('lon')

print('Helper functions defined ✓')


Helper functions defined ✓


## Section 5 — Fetch All Tag Groups

In [ ]:
import requests

test_query = '[out:json][timeout:10]; node["tourism"="museum"](40.75,-73.99,40.76,-73.98); out body;'

r = requests.post(
    'https://overpass-api.de/api/interpreter',
    data={'data': test_query},
    timeout=30,
    headers={'User-Agent': 'GemFinder-Research/1.0'}
)

print('Status:', r.status_code)
print('Response:', r.text[:500])  # inspect error messages if any


Status: 200
Response: {
  "version": 0.6,
  "generator": "Overpass API 0.7.62.11 87bfad18",
  "osm3s": {
    "timestamp_osm_base": "2026-06-03T10:55:42Z",
    "copyright": "The data included in this document is from www.openstreetmap.org. The data is made available under ODbL."
  },
  "elements": [

{
  "type": "node",
  "id": 12917522000,
  "lat": 40.7575933,
  "lon": -73.9843375,
  "tags": {
    "check_date": "2025-06-15",
    "name": "Museum of Broadway",
    "tourism": "museum"
  }
}

  ]
}



In [ ]:
all_elements = []     # raw Overpass elements
seen_osm_ids = set()  # deduplication: (type, id)

for group_name, tag_filter in tqdm(TAG_GROUPS, desc='Querying Overpass'):
    query = build_query(tag_filter, BBOX_STR)
    elements = overpass_request(query)

    new_count = 0
    for el in elements:
        key = (el['type'], el['id'])
        if key not in seen_osm_ids:
            seen_osm_ids.add(key)
            el['_group'] = group_name  # record which tag group found this
            all_elements.append(el)
            new_count += 1

    print(f'  {group_name:25s} → {len(elements):4d} elements, {new_count:4d} new (total: {len(all_elements)})')
    time.sleep(REQUEST_DELAY_SEC)

print(f'\n✅ Total unique elements: {len(all_elements)}')

# Save raw JSON (useful for debugging and re-running without re-querying)
raw_path = OUTPUT_DIR / 'attractions_raw.json'
with open(raw_path, 'w', encoding='utf-8') as f:
    json.dump(all_elements, f, ensure_ascii=False, indent=2)
print(f'Raw JSON saved → {raw_path}')


Querying Overpass:   0%|          | 0/20 [00:00<?, ?it/s]

  HTTP 403 from https://maps.mail.ru/osm/tools/overpass/api/interpreter, retrying...
  museum                    →  107 elements,  107 new (total: 107)


Querying Overpass:   5%|▌         | 1/20 [00:11<03:35, 11.33s/it]

  gallery                   →  279 elements,  279 new (total: 386)


Querying Overpass:  10%|█         | 2/20 [00:16<02:14,  7.50s/it]

...

  HTTP 403 from https://maps.mail.ru/osm/tools/overpass/api/interpreter, retrying...
  Server busy (504) on https://overpass-api.de/api/interpreter, waiting 20s...
  Timeout on https://overpass.kumi.systems/api/interpreter (attempt 3), retrying...
  ⚠️ All 3 attempts failed, returning empty list
  historic_building         →    0 elements,    0 new (total: 8631)


Querying Overpass:  95%|█████████▌| 19/20 [21:36<01:47, 107.56s/it]

  Server busy (504) on https://overpass-api.de/api/interpreter, waiting 10s...
  HTTP 403 from https://maps.mail.ru/osm/tools/overpass/api/interpreter, retrying...
  Timeout on https://overpass.kumi.systems/api/interpreter (attempt 3), retrying...
  ⚠️ All 3 attempts failed, returning empty list
  landmark                  →    0 elements,    0 new (total: 8631)


Querying Overpass: 100%|██████████| 20/20 [23:45<00:00, 71.28s/it] 



✅ Total unique elements: 8631
Raw JSON saved → output/attractions_raw.json


## Section 6 — Clean Data & Map to Gem Finder Categories

In [102]:
# Reload all_elements from the raw JSON — no need to re-query Overpass
import json
with open(OUTPUT_DIR / 'attractions_raw.json', encoding='utf-8') as f:
    all_elements = json.load(f)
print(f'Loaded {len(all_elements)} elements from raw JSON')


Loaded 8631 elements from raw JSON


In [104]:
CATEGORY_RULES = [
    ('Museum',        'tourism',  {'museum', 'gallery', 'aquarium'}),
    ('Zoo',           'tourism',  {'zoo', 'theme_park'}),
    ('Park',          'leisure',  {'park', 'garden', 'nature_reserve'}),
    ('Stadium',       'leisure',  {'stadium'}),
    ('Entertainment', 'amenity',  {'theatre', 'cinema', 'arts_centre'}),
    ('Landmark',      'tourism',  {'attraction', 'viewpoint', 'artwork'}),
    ('Landmark',      'historic', {'monument', 'memorial', 'building', 'landmark'}),
    ('Culture',       'amenity',  {'place_of_worship'}),
]

def infer_category(tags: dict) -> str:
    for gem_cat, key, values in CATEGORY_RULES:
        if tags.get(key) in values:
            return gem_cat
    return 'Other'

records = []
skipped_no_name = 0
skipped_no_coords = 0

for el in all_elements:
    tags = el.get('tags', {})
    name = tags.get('name', '').strip()

    if not name:
        skipped_no_name += 1
        continue

    lat, lon = extract_center(el)
    if lat is None or lon is None:
        skipped_no_coords += 1
        continue

    category = infer_category(tags)

    wheelchair_raw = tags.get('wheelchair', '').lower()
    wheelchair = wheelchair_raw if wheelchair_raw in ('yes', 'limited', 'no') else 'no'

    records.append({
        'osm_id':               f"{el['type']}/{el['id']}",
        'name':                 name,
        'name_en':              tags.get('name:en', ''),
        'category':             category,
        'osm_group':            el.get('_group', ''),
        'lat':                  lat,
        'lon':                  lon,
        'opening_hours':        tags.get('opening_hours', ''),   # raw OSM value; empty if absent
        'opening_hours_source': 'osm' if tags.get('opening_hours') else 'missing',
        'wheelchair':           wheelchair,
        'website':              tags.get('website') or tags.get('contact:website', ''),
        'phone':                tags.get('phone') or tags.get('contact:phone', ''),
        'addr_street':          tags.get('addr:street', ''),
        'addr_housenumber':     tags.get('addr:housenumber', ''),
        'description':          tags.get('description', ''),
        'avg_rating':           None,
        'rating_count':         None,
        'image_url':            None,
        'zone_id':              None,
    })

df = pd.DataFrame(records)
print(f'Total records    : {len(all_elements)}')
print(f'Skipped no name  : {skipped_no_name}')
print(f'Skipped no coords: {skipped_no_coords}')
print(f'Valid attractions: {len(df)}')
print()
print('Category breakdown:')
print(df['category'].value_counts().to_string())


Total records    : 8631
Skipped no name  : 5771
Skipped no coords: 0
Valid attractions: 2860

Category breakdown:
category
Park             936
Culture          753
Landmark         739
Museum           383
Entertainment     45
Zoo                3
Stadium            1


## Section 7 — Data Quality Check & Filtering

In [106]:
print('=== Data quality check ===')
print(f'Opening hours has value  : {df["opening_hours"].notna().sum()} / {len(df)}')
print(f'Opening hours from OSM   : {(df["opening_hours_source"]=="osm").sum()}')
print(f'Opening hours (default)  : {(df["opening_hours_source"]=="default").sum()}')
print()
print('Wheelchair breakdown:')
print(df['wheelchair'].value_counts().to_string())
print()
print('Sample records (first 5):')
display(df[['name', 'category', 'osm_group', 'lat', 'lon', 'wheelchair', 'opening_hours']].head())

# ── Optional: drop 'Other' rows with no useful information ───────
# Uncomment if 'Other' count is too large
# df = df[df['category'] != 'Other'].reset_index(drop=True)
# print(f'After filtering Other: {len(df)} records')

# ── Check for duplicate names ─────────────────────────────────────
dup_names = df[df.duplicated('name', keep=False)].sort_values('name')
if len(dup_names):
    print(f'\n⚠️  Duplicate names found ({len(dup_names)} rows):')
    display(dup_names[['osm_id', 'name', 'category', 'lat', 'lon']].head(20))
else:
    print('\n✅ No duplicate names')


=== Data quality check ===
Opening hours has value  : 2860 / 2860
Opening hours from OSM   : 373
Opening hours (default)  : 0

Wheelchair breakdown:
wheelchair
no         2691
yes         146
limited      23

Sample records (first 5):


,name,category,osm_group,lat,lon,wheelchair,opening_hours
0,Fort Lee Museum at the Judge Moore House,Museum,museum,40.848393,-73.968789,no,
1,Fraunces Tavern Museum,Museum,museum,40.703387,-74.011354,yes,12:00-17:00
2,National Museum of the American Indian,Museum,museum,40.704027,-74.013747,yes,Mo-Su 10:00-17:00; Dec 25 off
3,Bronx Museum of the Arts,Museum,museum,40.831091,-73.919947,yes,We-Su 13:00-18:00
4,Yeshiva University Museum,Museum,museum,40.737978,-73.993831,yes,



⚠️  Duplicate names found (308 rows):


,osm_id,name,category,lat,lon
568,node/2727025835,Alexander Hamilton,Landmark,40.781033,-73.964572
592,node/3846182257,Alexander Hamilton,Landmark,40.806713,-73.961776
2610,node/2837054155,Alexander Hamilton,Landmark,40.770212,-74.016916
2443,way/279622727,All Saints Roman Catholic Church,Culture,40.700970,-73.944571
2334,way/271610663,All Saints Roman Catholic Church,Culture,40.808267,-73.938232
361,node/12847512614,Amant,Museum,40.711852,-73.936819
353,node/12269996789,Amant,Museum,40.712231,-73.936987
2805,way/1146402190,Amsterdam Avenue,Landmark,40.838848,-73.937653
2806,way/1146402191,Amsterdam Avenue,Landmark,40.839133,-73.937453
1460,way/1280023786,Bankside,Park,40.807395,-73.930916


In [107]:
# Cell 7b — Deduplication
before = len(df)

# Prefer way > relation > node (way centroid coordinates are more accurate)
type_priority = {'way': 0, 'relation': 1, 'node': 2}
df['_type'] = df['osm_id'].str.split('/').str[0]
df['_type_rank'] = df['_type'].map(type_priority).fillna(3)

df = (
    df.sort_values('_type_rank')
      .drop_duplicates(subset=['name'], keep='first')
      .drop(columns=['_type', '_type_rank'])
      .reset_index(drop=True)
)

print(f'Before dedup : {before}')
print(f'After dedup  : {len(df)}')
print(f'Removed      : {before - len(df)} duplicate rows')


Before dedup : 2860
After dedup  : 2663
Removed      : 197 duplicate rows


## Section 8 — Generate Synthetic Rating Data

In [108]:
import numpy as np

np.random.seed(42)  # fixed seed for reproducibility

# ── Per-category rating parameters (reflect realistic user perception) ──
RATING_PARAMS = {
    'Museum':        {'mean': 4.3, 'std': 0.35, 'count_base': 2000},
    'Zoo':           {'mean': 4.4, 'std': 0.30, 'count_base': 3000},
    'Park':          {'mean': 4.5, 'std': 0.30, 'count_base': 3500},
    'Stadium':       {'mean': 4.2, 'std': 0.40, 'count_base': 1500},
    'Entertainment': {'mean': 4.0, 'std': 0.45, 'count_base': 1200},
    'Landmark':      {'mean': 4.1, 'std': 0.40, 'count_base': 1000},
    'Culture':       {'mean': 4.2, 'std': 0.35, 'count_base': 800},
    'Other':         {'mean': 3.8, 'std': 0.50, 'count_base': 500},
}

def synthetic_rating(category: str):
    params = RATING_PARAMS.get(category, RATING_PARAMS['Other'])
    rating = np.random.normal(params['mean'], params['std'])
    rating = round(float(np.clip(rating, 2.5, 5.0)), 1)
    # Use log-normal for review count (heavy-tailed, more realistic)
    count = int(np.random.lognormal(
        mean=np.log(params['count_base']),
        sigma=0.8
    ))
    count = max(count, 50)
    return rating, count

df[['avg_rating', 'rating_count']] = df['category'].apply(
    lambda c: pd.Series(synthetic_rating(c))
)

print('Rating distribution by category:')
print(df.groupby('category')[['avg_rating', 'rating_count']].agg(['mean', 'min', 'max']).round(2).to_string())


Rating distribution by category:
              avg_rating           rating_count                 
                    mean  min  max         mean     min      max
category                                                        
Culture             4.20  3.1  5.0      1071.81    63.0  10291.0
Entertainment       3.98  2.9  4.9      1503.86   243.0   6953.0
Landmark            4.10  3.0  5.0      1400.25    78.0  23126.0
Museum              4.28  3.3  5.0      2817.04   207.0  24614.0
Park                4.50  3.5  5.0      5128.57   319.0  76321.0
Stadium             5.00  5.0  5.0       812.00   812.0    812.0
Zoo                 4.23  4.2  4.3      2964.00  1043.0   4937.0


In [109]:
DEFAULT_HOURS = {
    'Museum':        'Tu-Su 10:00-17:00',
    'Zoo':           'Mo-Su 10:00-17:00',
    'Park':          '24/7',
    'Stadium':       None,
    'Entertainment': 'Mo-Su 12:00-23:00',
    'Landmark':      '24/7',
    'Culture':       'Mo-Su 08:00-20:00',
    'Other':         None,
}

# Back-fill missing opening_hours with category defaults
mask = df['opening_hours_source'] == 'missing'
df.loc[mask, 'opening_hours'] = df.loc[mask, 'category'].map(DEFAULT_HOURS)
df.loc[mask, 'opening_hours_source'] = 'default'

print('Opening hours source breakdown:')
print(df['opening_hours_source'].value_counts().to_string())
print(f'\nStill empty after backfill: {df["opening_hours"].isna().sum()} rows')


Opening hours source breakdown:
opening_hours_source
default    2309
osm         354

Still empty after backfill: 1 rows


In [110]:
SUGGESTED_DURATION = {
    'Museum':        120,
    'Zoo':           150,
    'Park':          60,
    'Stadium':       180,
    'Entertainment': 150,
    'Landmark':      30,
    'Culture':       45,
    'Other':         60,
}

df['suggested_duration'] = df['category'].map(SUGGESTED_DURATION)

print('Suggested duration by category (minutes):')
print(df.groupby('category')['suggested_duration'].first().to_string())
print(f'\n✅ suggested_duration written to df — {df["suggested_duration"].notna().sum()} rows')


Suggested duration by category (minutes):
category
Culture           45
Entertainment    150
Landmark          30
Museum           120
Park              60
Stadium          180
Zoo              150

✅ suggested_duration written to df — 2663 rows


## Section 9 — Spatial Join: Assign TLC Zone ID

In [111]:
# Requires the taxi_zones shapefile
# Download: https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip
#
# Install geopandas if missing:
# !pip install geopandas shapely

import urllib.request
import zipfile

SHAPEFILE_ZIP  = OUTPUT_DIR / 'taxi_zones.zip'
SHAPEFILE_DIR  = OUTPUT_DIR / 'taxi_zones'
SHAPEFILE_PATH = SHAPEFILE_DIR / 'taxi_zones' / 'taxi_zones.shp'

# Download shapefile if not already present
if not SHAPEFILE_PATH.exists():
    print('Downloading taxi_zones shapefile...')
    urllib.request.urlretrieve(
        'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip',
        SHAPEFILE_ZIP
    )
    with zipfile.ZipFile(SHAPEFILE_ZIP) as z:
        z.extractall(SHAPEFILE_DIR)
    print('✅ Shapefile downloaded and extracted')
else:
    print('Shapefile already exists, skipping download')

# Spatial join
try:
    import geopandas as gpd
    from shapely.geometry import Point

    zones_gdf = gpd.read_file(SHAPEFILE_PATH).to_crs(epsg=4326)

    attractions_gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(lon, lat) for lat, lon in zip(df['lat'], df['lon'])],
        crs='EPSG:4326'
    )

    joined = gpd.sjoin(attractions_gdf, zones_gdf[['LocationID', 'geometry']], how='left', predicate='within')
    df['zone_id'] = joined['LocationID'].values

    matched = df['zone_id'].notna().sum()
    print(f'Zone matched: {matched} / {len(df)} ({matched/len(df)*100:.1f}%)')
    print(f'Unmatched (zone_id=NaN): {len(df) - matched}')

except ImportError:
    print('⚠️  geopandas not installed — zone_id will be null')
    print('   Run: pip install geopandas shapely')
    print('   Then re-run this cell')


Shapefile already exists, skipping download
Zone matched: 2513 / 2663 (94.4%)
Unmatched (zone_id=NaN): 150


In [112]:
touches_mask = attractions_gdf.geometry.apply(
    lambda pt: zones_gdf.geometry.touches(pt).any()
)
print(f'Points falling on the boundary: {touches_mask.sum()}')

Points falling on the boundary: 0


In [113]:
df = df[df['zone_id'].notna()].reset_index(drop=True)
print(f'After removing the drift point: {len(df)} ')

After removing the drift point: 2513 


In [114]:
# Save CSV
csv_path = OUTPUT_DIR / 'attractions_process.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'✅ attractions.csv saved → {csv_path}  ({len(df)} rows)')

✅ attractions.csv saved → output/attractions_process.csv  (2513 rows)


In [115]:
csv_path = OUTPUT_DIR / 'attractions_process.csv'

df = pd.read_csv(csv_path, encoding='utf-8')
print(f'✅ attractions_process.csv loaded → {csv_path}  ({len(df)} rows)')

✅ attractions_process.csv loaded → output/attractions_process.csv  (2513 rows)


In [116]:
from dotenv import load_dotenv
import os

load_dotenv()
PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')


In [123]:
# Cell 9c — Download attraction images (Pexels API)
# Requires a free Pexels API key: https://www.pexels.com/api/

import re
import json
import time
import requests
from pathlib import Path
from tqdm import tqdm

# ── Config ──────────────────────────────────────────────────
# PEXELS_API_KEY = 'YOUR_PEXELS_API_KEY_HERE'
IMAGES_DIR      = OUTPUT_DIR / 'images'
SOURCE_MAP_PATH = IMAGES_DIR / 'source_map.json'
REQUEST_DELAY   = 19   # 200 req/hour = 1 req/18s; 19s to be safe

# ── Category fallback search queries ────────────────────────
CATEGORY_FALLBACK_QUERY = {
    'Museum':        'museum interior art gallery new york',
    'Zoo':           'zoo animals wildlife',
    'Park':          'central park new york green',
    'Stadium':       'stadium sports arena',
    'Entertainment': 'broadway theatre stage performance',
    'Landmark':      'new york landmark monument',
    'Culture':       'church cathedral architecture interior',
    'Other':         'new york city attraction',
}

# ── Helper functions ─────────────────────────────────────────

def slugify(name: str) -> str:
    name = name.lower()
    name = re.sub(r'[^a-z0-9]+', '-', name)
    name = name.strip('-')
    return name[:80]


def pexels_search(query: str, per_page: int = 1) -> list:
    url     = 'https://api.pexels.com/v1/search'
    headers = {'Authorization': PEXELS_API_KEY}
    params  = {'query': query, 'per_page': per_page, 'orientation': 'landscape'}
    try:
        r = requests.get(url, headers=headers, params=params, timeout=15)
        if r.status_code == 200:
            photos = r.json().get('photos', [])
            return [p['src']['large'] for p in photos]
        elif r.status_code == 429:
            print('  ⚠️  Rate limit hit, sleeping 3600s...')
            time.sleep(3600)
        else:
            print(f'  Pexels HTTP {r.status_code} for query: {query}')
    except Exception as e:
        print(f'  Pexels error: {e}')
    return []


def download_image(url: str, save_path: Path) -> bool:
    try:
        r = requests.get(url, timeout=30, stream=True)
        if r.status_code == 200:
            save_path.parent.mkdir(parents=True, exist_ok=True)
            with open(save_path, 'wb') as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            return True
    except Exception as e:
        print(f'  Download error: {e}')
    return False


def save_source_map(source_map: dict):
    """Persist source_map to disk so resume runs can restore true source values."""
    SOURCE_MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(SOURCE_MAP_PATH, 'w') as f:
        json.dump(source_map, f, ensure_ascii=False, indent=2)


# ── Load existing source_map (resume: restore source records from previous run) ──
if SOURCE_MAP_PATH.exists():
    with open(SOURCE_MAP_PATH) as f:
        source_map = json.load(f)
    print(f'Loaded source_map with {len(source_map)} existing records.')
else:
    source_map = {}
    print('No existing source_map found, starting fresh.')


# ── Step 1: Pre-fetch MULTIPLE fallback images per category ──
# Pre-fetch 15 fallback images per category so each backfill
# uses a distinct photo rather than repeating the same one.
FALLBACK_POOL_SIZE = 80

category_fallback_pool = {}   # cat → [rel_path, rel_path, ...]
category_fallback_idx  = {}   # cat → round-robin pointer (next index to use)

print('\nPre-fetching category fallback pools...')
for cat, query in CATEGORY_FALLBACK_QUERY.items():
    pool   = []
    fb_dir = IMAGES_DIR / cat.lower()

    # Resume from any images already downloaded in a previous run
    existing = sorted(fb_dir.glob('_fallback_*.jpg'))
    pool     = [str(p.relative_to(OUTPUT_DIR)) for p in existing]

    if len(pool) < FALLBACK_POOL_SIZE:
        urls = pexels_search(query, per_page=FALLBACK_POOL_SIZE)
        time.sleep(REQUEST_DELAY)

        downloaded_count = len(pool)
        for url in urls:
            if downloaded_count >= FALLBACK_POOL_SIZE:
                break
            fb_path = fb_dir / f'_fallback_{downloaded_count:03d}.jpg'
            if not fb_path.exists():
                if download_image(url, fb_path):
                    pool.append(str(fb_path.relative_to(OUTPUT_DIR)))
                    downloaded_count += 1
            else:
                pool.append(str(fb_path.relative_to(OUTPUT_DIR)))
                downloaded_count += 1

    category_fallback_pool[cat] = pool
    category_fallback_idx[cat]  = 0
    print(f'  {cat:15s} → {len(pool)} fallback images ready')

print(f'\nFallback pools ready.\n')


def get_next_fallback(category: str):
    """Return the next unused fallback image for the given category (round-robin)."""
    pool = category_fallback_pool.get(category, [])
    if not pool:
        return None
    idx  = category_fallback_idx[category]
    path = pool[idx % len(pool)]
    category_fallback_idx[category] += 1
    return path


# ── Step 2: Download one image per attraction ────────────────
image_paths   = []
image_sources = []
stats = {'pexels_name': 0, 'pexels_category': 0, 'failed': 0}

for _, row in tqdm(df.iterrows(), total=len(df), desc='Downloading images'):
    category  = row['category']
    slug      = slugify(row['name'])
    save_path = IMAGES_DIR / category.lower() / f'{slug}.jpg'
    rel_path  = str(save_path.relative_to(OUTPUT_DIR))

    # ── Resume: file already exists — read true source from source_map ──
    if save_path.exists():
        image_paths.append(rel_path)
        image_sources.append(source_map.get(slug, 'unknown'))
        continue

    # ── Strategy 1: Search by attraction name ──
    urls = pexels_search(f'{row["name"]} New York', per_page=1)
    time.sleep(REQUEST_DELAY)
    if urls and download_image(urls[0], save_path):
        source_map[slug] = 'pexels_name'
        save_source_map(source_map)
        image_paths.append(rel_path)
        image_sources.append('pexels_name')
        stats['pexels_name'] += 1
        continue

    # ── Strategy 2: Category fallback pool ──
    fb = get_next_fallback(category)
    if fb:
        source_map[slug] = 'pexels_category'
        save_source_map(source_map)
        image_paths.append(fb)
        image_sources.append('pexels_category')
        stats['pexels_category'] += 1
        continue

    # ── Strategy 3: All strategies exhausted ──
    source_map[slug] = 'failed'
    save_source_map(source_map)
    image_paths.append(None)
    image_sources.append('failed')
    stats['failed'] += 1

df['image_path']   = image_paths
df['image_source'] = image_sources

# ── Summary (newly downloaded this run only) ─────────────────
total_new = stats['pexels_name'] + stats['pexels_category'] + stats['failed']
print(f'\n=== Image download summary (this run) ===')
print(f'By attraction name : {stats["pexels_name"]}')
print(f'By category reuse  : {stats["pexels_category"]}')
print(f'Failed (None)      : {stats["failed"]}')
print(f'Total coverage     : {(len(df) - stats["failed"]) / len(df) * 100:.1f}%')
print(f'\nImages saved to    : {IMAGES_DIR.resolve()}')
print(f'Source map saved to: {SOURCE_MAP_PATH.resolve()}')

print(f'\n=== Source breakdown ===')
print(df['image_source'].value_counts().to_string())

fallback_df = df[df['image_source'] == 'pexels_category'][['name', 'category', 'image_path']]
if not fallback_df.empty:
    print(f'\n⚠️  Attractions using fallback images ({len(fallback_df)}):')
    print(fallback_df.to_string(index=False))

No existing source_map found, starting fresh.

Pre-fetching category fallback pools...
  Museum          → 80 fallback images ready
  Zoo             → 80 fallback images ready
  Park            → 80 fallback images ready
  Stadium         → 80 fallback images ready
  Entertainment   → 80 fallback images ready
  Landmark        → 80 fallback images ready
  Culture         → 80 fallback images ready
  Other           → 80 fallback images ready

Fallback pools ready.



  Download error: HTTPSConnectionPool(host='images.pexels.com', port=443): Max retries exceeded with url: /photos/5845467/pexels-photo-5845467.jpeg?auto=compress&cs=tinysrgb&h=650&w=940 (Caused by NameResolutionError("HTTPSConnection(host='images.pexels.com', port=443): Failed to resolve 'images.pexels.com' ([Errno 8] nodename nor servname provided, or not known)"))
  Pexels error: HTTPSConnectionPool(host='api.pexels.com', port=443): Max retries exceeded with url: /v1/search?query=New+York+City+Fire+Museum+New+York&per_page=1&orientation=landscape (Caused by NameResolutionError("HTTPSConnection(host='api.pexels.com', port=443): Failed to resolve 'api.pexels.com' ([Errno 8] nodename nor servname provided, or not known)"))


  Pexels error: HTTPSConnectionPool(host='api.pexels.com', port=443): Max retries exceeded with url: /v1/search?query=Neue+Galerie+New+York+New+York&per_page=1&orientation=landscape (Caused by NameResolutionError("HTTPSConnection(host='api.pexels.com', port=443): Failed to resolve 'api.pexels.com' ([Errno 8] nodename nor servname provided, or not known)"))


  Pexels error: HTTPSConnectionPool(host='api.pexels.com', port=443): Max retries exceeded with url: /v1/search?query=MoMA+New+York&per_page=1&orientation=landscape (Caused by NameResolutionError("HTTPSConnection(host='api.pexels.com', port=443): Failed to resolve 'api.pexels.com' ([Errno 8] nodename nor servname provided, or not known)"))


  Pexels HTTP 403 for query: Galleria Ca' d'Oro New York



=== Image download summary (this run) ===
By attraction name : 2503
By category reuse  : 6
Failed (None)      : 0
Total coverage     : 100.0%

Images saved to    : /Users/rj/Desktop/dev/code/COMP47360/Data/output/images
Source map saved to: /Users/rj/Desktop/dev/code/COMP47360/Data/output/images/source_map.json

=== Source breakdown ===
image_source
pexels_name        2507
pexels_category       6

⚠️  Attractions using fallback images (6):
                     name category                        image_path
     I♥NY More Than Ever! Landmark images/landmark/_fallback_000.jpg
Museum at Eldridge Street   Museum   images/museum/_fallback_000.jpg
New York City Fire Museum   Museum   images/museum/_fallback_001.jpg
    Neue Galerie New York   Museum   images/museum/_fallback_002.jpg
                     MoMA   Museum   images/museum/_fallback_003.jpg
       Galleria Ca' d'Oro   Museum   images/museum/_fallback_004.jpg


In [ ]:
# # Cell 9c — Download attraction images (Pexels API)
# # Requires a free Pexels API key: https://www.pexels.com/api/

# import re
# import time
# import requests
# from pathlib import Path
# from tqdm import tqdm

# # ── Config ──────────────────────────────────────────────────
# # PEXELS_API_KEY = 'YOUR_PEXELS_API_KEY_HERE'
# IMAGES_DIR    = OUTPUT_DIR / 'images'
# REQUEST_DELAY = 19   # 200 req/hour = 1 req/18s; 19s to be safe

# # ── Category fallback search queries ────────────────────────
# CATEGORY_FALLBACK_QUERY = {
#     'Museum':        'museum interior art gallery new york',
#     'Zoo':           'zoo animals wildlife',
#     'Park':          'central park new york green',
#     'Stadium':       'stadium sports arena',
#     'Entertainment': 'broadway theatre stage performance',
#     'Landmark':      'new york landmark monument',
#     'Culture':       'church cathedral architecture interior',
#     'Other':         'new york city attraction',
# }

# # ── Helper functions ─────────────────────────────────────────

# def slugify(name: str) -> str:
#     name = name.lower()
#     name = re.sub(r'[^a-z0-9]+', '-', name)
#     name = name.strip('-')
#     return name[:80]


# def pexels_search(query: str, per_page: int = 1) -> list:
#     url     = 'https://api.pexels.com/v1/search'
#     headers = {'Authorization': PEXELS_API_KEY}
#     params  = {'query': query, 'per_page': per_page, 'orientation': 'landscape'}
#     try:
#         r = requests.get(url, headers=headers, params=params, timeout=15)
#         if r.status_code == 200:
#             photos = r.json().get('photos', [])
#             return [p['src']['large'] for p in photos]
#         elif r.status_code == 429:
#             print('  ⚠️  Rate limit hit, sleeping 3600s...')
#             time.sleep(3600)
#         else:
#             print(f'  Pexels HTTP {r.status_code} for query: {query}')
#     except Exception as e:
#         print(f'  Pexels error: {e}')
#     return []


# def download_image(url: str, save_path: Path) -> bool:
#     try:
#         r = requests.get(url, timeout=30, stream=True)
#         if r.status_code == 200:
#             save_path.parent.mkdir(parents=True, exist_ok=True)
#             with open(save_path, 'wb') as f:
#                 for chunk in r.iter_content(8192):
#                     f.write(chunk)
#             return True
#     except Exception as e:
#         print(f'  Download error: {e}')
#     return False


# # ── Step 1: Pre-fetch MULTIPLE fallback images per category ──
# # Pre-fetch 15 fallback images per category so each backfill
# # uses a distinct photo rather than repeating the same one.
# FALLBACK_POOL_SIZE = 15

# category_fallback_pool = {}   # cat → [rel_path, rel_path, ...]
# category_fallback_idx  = {}   # cat → round-robin pointer (next index to use)

# print('Pre-fetching category fallback pools...')
# for cat, query in CATEGORY_FALLBACK_QUERY.items():
#     pool = []
#     fb_dir = IMAGES_DIR / cat.lower()

#     # Resume from any images already downloaded in a previous run
#     existing = sorted(fb_dir.glob('_fallback_*.jpg'))
#     pool = [str(p.relative_to(OUTPUT_DIR)) for p in existing]

#     if len(pool) < FALLBACK_POOL_SIZE:
#         need = FALLBACK_POOL_SIZE - len(pool)
#         urls = pexels_search(query, per_page=FALLBACK_POOL_SIZE)
#         time.sleep(REQUEST_DELAY)

#         # Only download the images we still need (identified by index in filename)
#         downloaded_count = len(pool)
#         for url in urls:
#             if downloaded_count >= FALLBACK_POOL_SIZE:
#                 break
#             fb_path = fb_dir / f'_fallback_{downloaded_count:03d}.jpg'
#             if not fb_path.exists():
#                 if download_image(url, fb_path):
#                     pool.append(str(fb_path.relative_to(OUTPUT_DIR)))
#                     downloaded_count += 1
#             else:
#                 pool.append(str(fb_path.relative_to(OUTPUT_DIR)))
#                 downloaded_count += 1

#     category_fallback_pool[cat] = pool
#     category_fallback_idx[cat]  = 0
#     print(f'  {cat:15s} → {len(pool)} fallback images ready')

# print(f'\nFallback pools ready.\n')


# def get_next_fallback(category: str):
#     """Return the next unused fallback image for the given category (round-robin)."""
#     pool = category_fallback_pool.get(category, [])
#     if not pool:
#         return None
#     idx  = category_fallback_idx[category]
#     path = pool[idx % len(pool)]
#     category_fallback_idx[category] += 1
#     return path


# # ── Step 2: Download one image per attraction ────────────────
# image_paths   = []
# image_sources = []   # Track the origin of each image
# stats = {'already_exists': 0, 'pexels_name': 0, 'pexels_category': 0, 'failed': 0}

# for _, row in tqdm(df.iterrows(), total=len(df), desc='Downloading images'):
#     category  = row['category']
#     slug      = slugify(row['name'])
#     save_path = IMAGES_DIR / category.lower() / f'{slug}.jpg'
#     rel_path  = str(save_path.relative_to(OUTPUT_DIR))

#     # ── Already downloaded — skip ──
#     if save_path.exists():
#         image_paths.append(rel_path)
#         image_sources.append('already_exists')
#         stats['already_exists'] += 1
#         continue

#     # ── Strategy 1: Search by attraction name (most accurate) ──
#     urls = pexels_search(f'{row["name"]} New York', per_page=1)
#     time.sleep(REQUEST_DELAY)
#     if urls and download_image(urls[0], save_path):
#         image_paths.append(rel_path)
#         image_sources.append('pexels_name')       # ✅ Exact match
#         stats['pexels_name'] += 1
#         continue

#     # ── Strategy 2: Pull the next distinct image from the category pool ──
#     fb = get_next_fallback(category)
#     if fb:
#         image_paths.append(fb)
#         image_sources.append('pexels_category')   # ⚠️ Fallback
#         stats['pexels_category'] += 1
#         continue

#     # ── Strategy 3: All strategies exhausted — no image available ──
#     image_paths.append(None)
#     image_sources.append('failed')                # ❌ No image
#     stats['failed'] += 1

# df['image_path']   = image_paths
# df['image_source'] = image_sources   # New column: image origin label

# # ── Summary ─────────────────────────────────────────────────
# print(f'\n=== Image download summary ===')
# print(f'Already existed    : {stats["already_exists"]}')
# print(f'By attraction name : {stats["pexels_name"]}')
# print(f'By category reuse  : {stats["pexels_category"]}')
# print(f'Failed (None)      : {stats["failed"]}')
# print(f'Coverage           : {(len(df) - stats["failed"]) / len(df) * 100:.1f}%')
# print(f'\nImages saved to : {IMAGES_DIR.resolve()}')

# # Breakdown by source
# print(f'\n=== Source breakdown ===')
# print(df['image_source'].value_counts().to_string())

# # Optional: list attractions that fell back to category images for manual review
# fallback_df = df[df['image_source'] == 'pexels_category'][['name', 'category', 'image_path']]
# if not fallback_df.empty:
#     print(f'\n⚠️  Attractions using fallback images ({len(fallback_df)}):')
#     print(fallback_df.to_string(index=False))

Pre-fetching category fallback pools...
  Museum          → 15 fallback images ready
  Zoo             → 15 fallback images ready
  Park            → 15 fallback images ready
  Stadium         → 15 fallback images ready
  Entertainment   → 15 fallback images ready
  Landmark        → 15 fallback images ready
  Culture         → 15 fallback images ready
  Other           → 15 fallback images ready

Fallback pools ready.



  Pexels HTTP 403 for query: Galleria Ca' d'Oro New York



=== Image download summary ===
Already existed    : 1475
By attraction name : 1036
By category reuse  : 2
Failed (None)      : 0
Coverage           : 100.0%

Images saved to : /Users/rj/Desktop/dev/code/COMP47360 /Data/output/images

=== Source breakdown ===
image_source
already_exists     1475
pexels_name        1036
pexels_category       2

⚠️  Attractions using fallback images (2):
                name category                        image_path
I♥NY More Than Ever! Landmark images/landmark/_fallback_000.jpg
  Galleria Ca' d'Oro   Museum   images/museum/_fallback_000.jpg


In [ ]:
# # Cell 9c — Download attraction images (Pexels API)
# # Requires a free Pexels API key: https://www.pexels.com/api/

# import re
# import time
# import requests
# from pathlib import Path
# from tqdm import tqdm

# # ── Config ──────────────────────────────────────────────────
# # PEXELS_API_KEY = 'YOUR_PEXELS_API_KEY_HERE'   # ← fill in your key
# IMAGES_DIR    = OUTPUT_DIR / 'images'
# REQUEST_DELAY = 19   # 200 req/hour = 1 req/18s; 19s to be safe

# # ── Category fallback search queries ────────────────────────
# CATEGORY_FALLBACK_QUERY = {
#     'Museum':        'museum interior art gallery new york',
#     'Zoo':           'zoo animals wildlife',
#     'Park':          'central park new york green',
#     'Stadium':       'stadium sports arena',
#     'Entertainment': 'broadway theatre stage performance',
#     'Landmark':      'new york landmark monument',
#     'Culture':       'church cathedral architecture interior',
#     'Other':         'new york city attraction',
# }

# # ── Helper functions ─────────────────────────────────────────

# def slugify(name: str) -> str:
#     """Convert an attraction name to a filesystem-safe lowercase string."""
#     name = name.lower()
#     name = re.sub(r'[^a-z0-9]+', '-', name)
#     name = name.strip('-')
#     return name[:80]


# def pexels_search(query: str, per_page: int = 1) -> list:
#     """Search Pexels and return a list of large-size image URLs."""
#     url     = 'https://api.pexels.com/v1/search'
#     headers = {'Authorization': PEXELS_API_KEY}
#     params  = {'query': query, 'per_page': per_page, 'orientation': 'landscape'}
#     try:
#         r = requests.get(url, headers=headers, params=params, timeout=15)
#         if r.status_code == 200:
#             photos = r.json().get('photos', [])
#             return [p['src']['large'] for p in photos]
#         elif r.status_code == 429:
#             print('  ⚠️  Rate limit hit, sleeping 3600s...')
#             time.sleep(3600)
#         else:
#             print(f'  Pexels HTTP {r.status_code} for query: {query}')
#     except Exception as e:
#         print(f'  Pexels error: {e}')
#     return []


# def download_image(url: str, save_path: Path) -> bool:
#     """Download an image to disk. Returns True on success."""
#     try:
#         r = requests.get(url, timeout=30, stream=True)
#         if r.status_code == 200:
#             save_path.parent.mkdir(parents=True, exist_ok=True)
#             with open(save_path, 'wb') as f:
#                 for chunk in r.iter_content(8192):
#                     f.write(chunk)
#             return True
#     except Exception as e:
#         print(f'  Download error: {e}')
#     return False


# # ── Step 1: Pre-fetch one fallback image per category (8 API calls total) ──
# category_fallback_path = {}
# print('Pre-fetching category fallback images...')
# for cat, query in CATEGORY_FALLBACK_QUERY.items():
#     fb_path = IMAGES_DIR / cat.lower() / '_fallback_000.jpg'
#     fb_rel  = str(fb_path.relative_to(OUTPUT_DIR))
#     if not fb_path.exists():
#         urls = pexels_search(query, per_page=1)
#         time.sleep(REQUEST_DELAY)
#         if urls:
#             download_image(urls[0], fb_path)
#     category_fallback_path[cat] = fb_rel if fb_path.exists() else None
#     print(f'  {cat:15s} → {category_fallback_path[cat]}')

# print(f'\nFallback images ready: {sum(v is not None for v in category_fallback_path.values())} / {len(CATEGORY_FALLBACK_QUERY)}')


# # ── Step 2: Download one image per attraction (1 API call each) ─────────
# image_paths = []
# stats = {'already_exists': 0, 'pexels_name': 0, 'pexels_category': 0, 'failed': 0}

# for _, row in tqdm(df.iterrows(), total=len(df), desc='Downloading images'):
#     category  = row['category']
#     slug      = slugify(row['name'])
#     save_path = IMAGES_DIR / category.lower() / f'{slug}.jpg'
#     rel_path  = str(save_path.relative_to(OUTPUT_DIR))

#     # Already downloaded — skip (no API call)
#     if save_path.exists():
#         image_paths.append(rel_path)
#         stats['already_exists'] += 1
#         continue

#     # Strategy 1: search by attraction name
#     urls = pexels_search(f'{row["name"]} New York', per_page=1)
#     time.sleep(REQUEST_DELAY)
#     if urls and download_image(urls[0], save_path):
#         image_paths.append(rel_path)
#         stats['pexels_name'] += 1
#         continue

#     # Strategy 2: reuse pre-fetched category fallback (no extra API call)
#     fb = category_fallback_path.get(category)
#     if fb:
#         image_paths.append(fb)
#         stats['pexels_category'] += 1
#         continue

#     # Strategy 3: no image found
#     image_paths.append(None)
#     stats['failed'] += 1

# df['image_path'] = image_paths

# print(f'\n=== Image download summary ===')
# print(f'Already existed    : {stats["already_exists"]}')
# print(f'By attraction name : {stats["pexels_name"]}')
# print(f'By category reuse  : {stats["pexels_category"]}')
# print(f'Failed (None)      : {stats["failed"]}')
# print(f'Coverage           : {(len(df) - stats["failed"]) / len(df) * 100:.1f}%')
# print(f'\nImages saved to : {IMAGES_DIR.resolve()}')
# print(f'Example rel path: {df["image_path"].dropna().iloc[0] if df["image_path"].notna().any() else "N/A"}')


Pre-fetching category fallback images...
  Museum          → images/museum/_fallback_000.jpg
  Zoo             → images/zoo/_fallback_000.jpg
  Park            → images/park/_fallback_000.jpg
  Stadium         → images/stadium/_fallback_000.jpg
  Entertainment   → images/entertainment/_fallback_000.jpg
  Landmark        → images/landmark/_fallback_000.jpg
  Culture         → images/culture/_fallback_000.jpg
  Other           → images/other/_fallback_000.jpg

Fallback images ready: 8 / 8


  Pexels HTTP 403 for query: Galleria Ca' d'Oro New York



=== Image download summary ===
Already existed    : 2508
By attraction name : 3
By category reuse  : 2
Failed (None)      : 0
Coverage           : 100.0%

Images saved to : /Users/rj/Desktop/dev/code/COMP47360 /Data/output/images
Example rel path: images/park/dante-park.jpg


## Section 10 — Save Final Results

In [124]:
# Save CSV
csv_path = OUTPUT_DIR / 'attractions.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'✅ attractions.csv saved → {csv_path}  ({len(df)} rows)')

# Save JSON (for Spring Boot data import)
json_path = OUTPUT_DIR / 'attractions.json'
df.to_json(json_path, orient='records', force_ascii=False, indent=2)
print(f'✅ attractions.json saved → {json_path}')

# Final summary
print(f'\n=== Final dataset overview ===')
print(f'Total records: {len(df)}')
print(f'\nCategory breakdown:')
print(df['category'].value_counts().to_string())
print(f'\nWheelchair breakdown:')
print(df['wheelchair'].value_counts().to_string())
print(f'\nOpening hours source:')
print(df['opening_hours_source'].value_counts().to_string())
print(f'\nZone ID coverage: {df["zone_id"].notna().sum()} / {len(df)}')
print(f'Image coverage  : {df["image_path"].notna().sum()} / {len(df)}')
print()
display(df[['name', 'category', 'lat', 'lon', 'wheelchair', 'avg_rating', 'rating_count', 'zone_id', 'image_path']].head(10))


✅ attractions.csv saved → output/attractions.csv  (2513 rows)
✅ attractions.json saved → output/attractions.json

=== Final dataset overview ===
Total records: 2513

Category breakdown:
category
Park             851
Culture          657
Landmark         601
Museum           359
Entertainment     42
Zoo                2
Stadium            1

Wheelchair breakdown:
wheelchair
no         2359
yes         131
limited      23

Opening hours source:
opening_hours_source
default    2161
osm         352

Zone ID coverage: 2513 / 2513
Image coverage  : 2513 / 2513



,name,category,lat,lon,wheelchair,avg_rating,rating_count,zone_id,image_path
0,Dante Park,Park,40.771919,-73.982571,yes,4.7,11836.0,142.0,images/park/dante-park.jpg
1,Maria Hernandez Park,Park,40.703176,-73.923864,no,4.4,2902.0,36.0,images/park/maria-hernandez-park.jpg
2,Hamilton Fish Park,Park,40.719793,-73.980833,no,5.0,6466.0,232.0,images/park/hamilton-fish-park.jpg
3,Duane Park,Park,40.717163,-74.009407,no,4.4,5402.0,231.0,images/park/duane-park.jpg
4,Joseph Slifka Park,Park,40.743948,-73.972918,no,4.4,2411.0,233.0,images/park/joseph-slifka-park.jpg
5,Dutch Kills Green,Park,40.749708,-73.937001,no,4.6,757.0,146.0,images/park/dutch-kills-green.jpg
6,Queensboro Bridge Greenway,Park,40.750315,-73.939079,no,4.0,2232.0,146.0,images/park/queensboro-bridge-greenway.jpg
7,John Downing Park,Park,40.744550,-73.913842,no,4.2,4500.0,226.0,images/park/john-downing-park.jpg
8,Lance Corporal Thomas P Noonan Playground,Park,40.741061,-73.922465,no,4.2,1130.0,226.0,images/park/lance-corporal-thomas-p-noonan-pla...
9,Richard Tucker Square,Park,40.773442,-73.981889,no,4.9,2921.0,142.0,images/park/richard-tucker-square.jpg


In [126]:
import pandas as pd
pd.set_option('display.max_columns', None)
display(df.head(10))

,osm_id,name,category,osm_group,lat,lon,opening_hours,opening_hours_source,wheelchair,avg_rating,rating_count,zone_id,suggested_duration,image_path,image_source
0,way/154416663,Dante Park,Park,park,40.771919,-73.982571,24/7,default,yes,4.7,11836.0,142.0,60,images/park/dante-park.jpg,pexels_name
1,way/154374795,Maria Hernandez Park,Park,park,40.703176,-73.923864,24/7,default,no,4.4,2902.0,36.0,60,images/park/maria-hernandez-park.jpg,pexels_name
2,way/147475917,Hamilton Fish Park,Park,park,40.719793,-73.980833,24/7,default,no,5.0,6466.0,232.0,60,images/park/hamilton-fish-park.jpg,pexels_name
3,way/147035202,Duane Park,Park,park,40.717163,-74.009407,Mo-Su 06:00-01:00,osm,no,4.4,5402.0,231.0,60,images/park/duane-park.jpg,pexels_name
4,way/144333814,Joseph Slifka Park,Park,park,40.743948,-73.972918,24/7,default,no,4.4,2411.0,233.0,60,images/park/joseph-slifka-park.jpg,pexels_name
5,way/143932286,Dutch Kills Green,Park,park,40.749708,-73.937001,24/7,default,no,4.6,757.0,146.0,60,images/park/dutch-kills-green.jpg,pexels_name
6,way/143932285,Queensboro Bridge Greenway,Park,park,40.750315,-73.939079,24/7,default,no,4.0,2232.0,146.0,60,images/park/queensboro-bridge-greenway.jpg,pexels_name
7,way/143928486,John Downing Park,Park,park,40.744550,-73.913842,24/7,default,no,4.2,4500.0,226.0,60,images/park/john-downing-park.jpg,pexels_name
8,way/143928485,Lance Corporal Thomas P Noonan Playground,Park,park,40.741061,-73.922465,06:00-21:00,osm,no,4.2,1130.0,226.0,60,images/park/lance-corporal-thomas-p-noonan-pla...,pexels_name
9,way/154416666,Richard Tucker Square,Park,park,40.773442,-73.981889,24/7,default,no,4.9,2921.0,142.0,60,images/park/richard-tucker-square.jpg,pexels_name


In [127]:
import pandas as pd
pd.set_option('display.max_columns', None)
display(df.head(10))

,osm_id,name,category,osm_group,lat,lon,opening_hours,opening_hours_source,wheelchair,avg_rating,rating_count,zone_id,suggested_duration,image_path,image_source
0,way/154416663,Dante Park,Park,park,40.771919,-73.982571,24/7,default,yes,4.7,11836.0,142.0,60,images/park/dante-park.jpg,pexels_name
1,way/154374795,Maria Hernandez Park,Park,park,40.703176,-73.923864,24/7,default,no,4.4,2902.0,36.0,60,images/park/maria-hernandez-park.jpg,pexels_name
2,way/147475917,Hamilton Fish Park,Park,park,40.719793,-73.980833,24/7,default,no,5.0,6466.0,232.0,60,images/park/hamilton-fish-park.jpg,pexels_name
3,way/147035202,Duane Park,Park,park,40.717163,-74.009407,Mo-Su 06:00-01:00,osm,no,4.4,5402.0,231.0,60,images/park/duane-park.jpg,pexels_name
4,way/144333814,Joseph Slifka Park,Park,park,40.743948,-73.972918,24/7,default,no,4.4,2411.0,233.0,60,images/park/joseph-slifka-park.jpg,pexels_name
5,way/143932286,Dutch Kills Green,Park,park,40.749708,-73.937001,24/7,default,no,4.6,757.0,146.0,60,images/park/dutch-kills-green.jpg,pexels_name
6,way/143932285,Queensboro Bridge Greenway,Park,park,40.750315,-73.939079,24/7,default,no,4.0,2232.0,146.0,60,images/park/queensboro-bridge-greenway.jpg,pexels_name
7,way/143928486,John Downing Park,Park,park,40.744550,-73.913842,24/7,default,no,4.2,4500.0,226.0,60,images/park/john-downing-park.jpg,pexels_name
8,way/143928485,Lance Corporal Thomas P Noonan Playground,Park,park,40.741061,-73.922465,06:00-21:00,osm,no,4.2,1130.0,226.0,60,images/park/lance-corporal-thomas-p-noonan-pla...,pexels_name
9,way/154416666,Richard Tucker Square,Park,park,40.773442,-73.981889,24/7,default,no,4.9,2921.0,142.0,60,images/park/richard-tucker-square.jpg,pexels_name


In [128]:
# Save attractions_final (excluding Entertainment, Zoo, Stadium)
df_final = df[~df['category'].isin(['Entertainment', 'Zoo', 'Stadium'])].reset_index(drop=True)

csv_final_path = OUTPUT_DIR / 'attractions_v2.csv'
df_final.to_csv(csv_final_path, index=False, encoding='utf-8')
print(f'✅ attractions_v2.csv saved → {csv_final_path}  ({len(df_final)} rows)')

json_final_path = OUTPUT_DIR / 'attractions_v2.json'
df_final.to_json(json_final_path, orient='records', force_ascii=False, indent=2)
print(f'✅ attractions_final.json saved → {json_final_path}')

print(f'\n=== Final dataset overview ===')
print(f'Original : {len(df)} rows')
print(f'Filtered : {len(df_final)} rows (removed {len(df) - len(df_final)})')
print(f'\nCategory breakdown:')
print(df_final['category'].value_counts().to_string())
print(f'\nWheelchair breakdown:')
print(df_final['wheelchair'].value_counts().to_string())
print(f'\nOpening hours source:')
print(df_final['opening_hours_source'].value_counts().to_string())
print(f'\nZone ID coverage: {df_final["zone_id"].notna().sum()} / {len(df_final)}')
print(f'Image coverage  : {df_final["image_path"].notna().sum()} / {len(df_final)}')
print()
display(df_final[['name', 'category', 'lat', 'lon', 'wheelchair', 'avg_rating', 'rating_count', 'zone_id', 'image_path']].head(10))

✅ attractions_v2.csv saved → output/attractions_v2.csv  (2468 rows)
✅ attractions_final.json saved → output/attractions_v2.json

=== Final dataset overview ===
Original : 2513 rows
Filtered : 2468 rows (removed 45)

Category breakdown:
category
Park        851
Culture     657
Landmark    601
Museum      359

Wheelchair breakdown:
wheelchair
no         2327
yes         122
limited      19

Opening hours source:
opening_hours_source
default    2123
osm         345

Zone ID coverage: 2468 / 2468
Image coverage  : 2468 / 2468



,name,category,lat,lon,wheelchair,avg_rating,rating_count,zone_id,image_path
0,Dante Park,Park,40.771919,-73.982571,yes,4.7,11836.0,142.0,images/park/dante-park.jpg
1,Maria Hernandez Park,Park,40.703176,-73.923864,no,4.4,2902.0,36.0,images/park/maria-hernandez-park.jpg
2,Hamilton Fish Park,Park,40.719793,-73.980833,no,5.0,6466.0,232.0,images/park/hamilton-fish-park.jpg
3,Duane Park,Park,40.717163,-74.009407,no,4.4,5402.0,231.0,images/park/duane-park.jpg
4,Joseph Slifka Park,Park,40.743948,-73.972918,no,4.4,2411.0,233.0,images/park/joseph-slifka-park.jpg
5,Dutch Kills Green,Park,40.749708,-73.937001,no,4.6,757.0,146.0,images/park/dutch-kills-green.jpg
6,Queensboro Bridge Greenway,Park,40.750315,-73.939079,no,4.0,2232.0,146.0,images/park/queensboro-bridge-greenway.jpg
7,John Downing Park,Park,40.744550,-73.913842,no,4.2,4500.0,226.0,images/park/john-downing-park.jpg
8,Lance Corporal Thomas P Noonan Playground,Park,40.741061,-73.922465,no,4.2,1130.0,226.0,images/park/lance-corporal-thomas-p-noonan-pla...
9,Richard Tucker Square,Park,40.773442,-73.981889,no,4.9,2921.0,142.0,images/park/richard-tucker-square.jpg


In [129]:
# ── Data Quality Check: df_final (post-filter, final attractions dataset) ────

print('=' * 65)
print('ATTRACTIONS (df_final) — DATA QUALITY CHECK')
print('=' * 65)

# ── 1. Shape ─────────────────────────────────────────────────────────────────
print(f'\n[1] Shape: {df_final.shape}')
print(f'    Columns: {df_final.columns.tolist()}')

# ── 2. Missing values ────────────────────────────────────────────────────────
print(f'\n[2] Missing values:')
missing     = df_final.isnull().sum()
missing_pct = (missing / len(df_final) * 100).round(1)
missing_df  = pd.DataFrame({'count': missing, '%': missing_pct})
has_missing = missing_df[missing_df['count'] > 0]
if has_missing.empty:
    print('    No missing values ✓')
else:
    print(has_missing.to_string())

# ── 3. Duplicate attraction names ────────────────────────────────────────────
print(f'\n[3] Duplicate attraction names:')
dup_names = df_final[df_final.duplicated(subset=['name'], keep=False)] \
              .sort_values('name')[['name', 'category', 'lat', 'lon']]
if dup_names.empty:
    print('    No duplicate names ✓')
else:
    print(f'    WARNING: {dup_names["name"].nunique()} names appear more than once')
    print(dup_names.head(10).to_string(index=False))

# ── 4. Category breakdown ────────────────────────────────────────────────────
print(f'\n[4] Category breakdown:')
cat_counts = df_final['category'].value_counts()
for cat, n in cat_counts.items():
    print(f'    {cat:15s}: {n:4d}  ({n/len(df_final)*100:.1f}%)')

# ── 5. Coordinate sanity (Manhattan bbox) ────────────────────────────────────
print(f'\n[5] Coordinate sanity check (Manhattan bbox):')
S, W, N, E = MANHATTAN_BBOX
out_of_bbox = df_final[
    (df_final['lat'] < S) | (df_final['lat'] > N) |
    (df_final['lon'] < W) | (df_final['lon'] > E)
]
print(f'    Lat range : {df_final["lat"].min():.5f} ~ {df_final["lat"].max():.5f}')
print(f'    Lon range : {df_final["lon"].min():.5f} ~ {df_final["lon"].max():.5f}')
if out_of_bbox.empty:
    print('    All coords within Manhattan bbox ✓')
else:
    print(f'    WARNING: {len(out_of_bbox)} attractions outside bbox:')
    print(out_of_bbox[['name', 'category', 'lat', 'lon']].to_string(index=False))

# ── 6. Opening hours coverage ─────────────────────────────────────────────────
print(f'\n[6] Opening hours coverage:')
print(df_final['opening_hours_source'].value_counts().to_string())
osm_pct = (df_final['opening_hours_source'] == 'osm').sum() / len(df_final) * 100
print(f'    OSM coverage: {osm_pct:.1f}%'
      + ('  ✓' if osm_pct > 10 else '  ← LOW — consider adding defaults'))

# ── 7. Rating sanity ──────────────────────────────────────────────────────────
print(f'\n[7] Rating sanity check:')
print(f'    avg_rating   — min: {df_final["avg_rating"].min():.2f}  '
      f'max: {df_final["avg_rating"].max():.2f}  '
      f'mean: {df_final["avg_rating"].mean():.2f}')
print(f'    rating_count — min: {df_final["rating_count"].min():.0f}  '
      f'max: {df_final["rating_count"].max():.0f}  '
      f'median: {df_final["rating_count"].median():.0f}')
invalid_rating = df_final[(df_final['avg_rating'] < 1) | (df_final['avg_rating'] > 5)]
print(f'    Out-of-range avg_rating (< 1 or > 5): {len(invalid_rating)}'
      + (' ✓' if invalid_rating.empty else '  ← WARNING'))

# ── 8. Zone ID coverage ───────────────────────────────────────────────────────
print(f'\n[8] Zone ID coverage:')
zone_ok      = df_final['zone_id'].notna().sum()
zone_missing = df_final['zone_id'].isna().sum()
print(f'    Assigned : {zone_ok}  ({zone_ok/len(df_final)*100:.1f}%)')
print(f'    Missing  : {zone_missing}'
      + (' ✓' if zone_missing == 0 else
         '  ← WARNING: these attractions cannot be linked to busyness data'))
if zone_missing > 0:
    print(df_final[df_final['zone_id'].isna()][['name', 'category', 'lat', 'lon']]
          .head(5).to_string(index=False))

# ── 9. Image coverage & source breakdown (updated for new Cell 9c logic) ──────
print(f'\n[9] Image coverage & source:')
img_ok      = df_final['image_path'].notna().sum()
img_missing = df_final['image_path'].isna().sum()
print(f'    Total with image : {img_ok}  ({img_ok/len(df_final)*100:.1f}%)')
print(f'    No image (failed): {img_missing}'
      + (' ✓' if img_missing == 0 else '  ← will need fallback at API serving time'))

if 'image_source' in df_final.columns:
    print(f'\n    Source breakdown:')
    for src, n in df_final['image_source'].value_counts().items():
        pct = n / len(df_final) * 100
        icons = {'pexels_name': '✅', 'already_exists': '✅',
                 'pexels_category': '⚠️ ', 'failed': '❌'}
        icon = icons.get(src, '  ')
        print(f'    {icon} {src:20s}: {n:5d}  ({pct:.1f}%)')

    # List fallback attractions for manual review
    fallback_rows = df_final[df_final['image_source'] == 'pexels_category']
    if not fallback_rows.empty:
        print(f'\n    ⚠️  Fallback images in use ({len(fallback_rows)}) '
              f'— consider manual replacement:')
        print(fallback_rows[['name', 'category', 'image_path']]
              .head(10).to_string(index=False))
        if len(fallback_rows) > 10:
            print(f'    ... and {len(fallback_rows) - 10} more')
else:
    print('    image_source column not found — re-run Cell 9c to generate it')

# ── 10. Wheelchair accessibility ──────────────────────────────────────────────
print(f'\n[10] Wheelchair accessibility:')
print(df_final['wheelchair'].value_counts().to_string())

print('\n' + '=' * 65)
print('CHECK COMPLETE')
print('=' * 65)

ATTRACTIONS (df_final) — DATA QUALITY CHECK

[1] Shape: (2468, 15)
    Columns: ['osm_id', 'name', 'category', 'osm_group', 'lat', 'lon', 'opening_hours', 'opening_hours_source', 'wheelchair', 'avg_rating', 'rating_count', 'zone_id', 'suggested_duration', 'image_path', 'image_source']

[2] Missing values:
    No missing values ✓

[3] Duplicate attraction names:
    No duplicate names ✓

[4] Category breakdown:
    Park           :  851  (34.5%)
    Culture        :  657  (26.6%)
    Landmark       :  601  (24.4%)
    Museum         :  359  (14.5%)

[5] Coordinate sanity check (Manhattan bbox):
    Lat range : 40.69519 ~ 40.88220
    Lon range : -74.01860 ~ -73.90560
                                      name category       lat        lon
                         Taaffe Playground     Park 40.695187 -73.960722
                                Devoe Park     Park 40.863537 -73.906554
                         Isla Verde Garden     Park 40.814931 -73.906823
                 Korean War Veter

In [130]:
# ── Remove attractions outside Manhattan bounding box ────────────────────────
S, W, N, E = MANHATTAN_BBOX

before = len(df_final)

# Keep only rows where coordinates fall within the bbox
df_final = df_final[
    (df_final['lat'] >= S) & (df_final['lat'] <= N) &
    (df_final['lon'] >= W) & (df_final['lon'] <= E)
].reset_index(drop=True)

after = len(df_final)

print(f'Removed : {before - after} out-of-bbox attractions')
print(f'Remaining: {after}')

Removed : 9 out-of-bbox attractions
Remaining: 2459


In [131]:
# Quick sanity check after bbox filtering
assert df_final['lat'].between(S, N).all() and df_final['lon'].between(W, E).all(), \
    "Still has out-of-bbox points!"

print(f'All {len(df_final)} attractions within Manhattan bbox ✓')

All 2459 attractions within Manhattan bbox ✓


In [133]:
# Save CSV
csv_path = OUTPUT_DIR / 'attractions_final.csv'
df_final.to_csv(csv_path, index=False, encoding='utf-8')
print(f'✅ attractions_final.csv saved → {csv_path}  ({len(df_final)} rows)')

# Save JSON (for Spring Boot data import)
json_path = OUTPUT_DIR / 'attractions_final.json'
df_final.to_json(json_path, orient='records', force_ascii=False, indent=2)
print(f'✅ attractions_final.json saved → {json_path}')

✅ attractions_final.csv saved → output/attractions_final.csv  (2459 rows)
✅ attractions_final.json saved → output/attractions_final.json
